In [151]:
import pandas as pd
from scipy.stats import binom
df = pd.read_pickle('trips.pkl')
print(df.shape)

(2486, 9)


In [152]:
def distance_km(p1, p2):
    d_lat = (p2['lat'] - p1['lat']) * 111
    d_lon = (p2['lon'] - p1['lon']) * 111
    return (d_lat ** 2 + d_lon ** 2) ** 0.5

In [153]:
def net_displacement_km(trace):
    return distance_km(trace[0], trace[-1])

In [154]:
df['displacement_km'] = df['gps_trace'].apply(net_displacement_km)
print(df.groupby('_true_label')['displacement_km'].describe())

                      count       mean       std       min       25%  \
_true_label                                                            
completed            2164.0  11.308462  5.769203  1.701369  6.375285   
fraud_cancel_naive     40.0  11.249273  5.363417  2.379362  6.653666   
fraud_cancel_silent    26.0  10.961070  4.983429  3.481600  7.352951   
genuine_cancel        256.0   2.705916  1.175557  0.757363  1.721087   

                           50%        75%        max  
_true_label                                           
completed            11.012704  15.492881  26.325602  
fraud_cancel_naive   11.042752  14.497525  20.845882  
fraud_cancel_silent  10.882309  14.034898  21.476818  
genuine_cancel        2.591111   3.524473   5.826626  


In [155]:
def km_after_cancel(row):
    if pd.isna(row['cancel_time']):
        return None
    after = [p for p in row['gps_trace'] if p['time'] >= row['cancel_time']]
    if not after:
        return 0.0
    return distance_km(after[0], after[-1])

In [156]:
df['km_after_cancel'] = df.apply(km_after_cancel, axis=1)

In [157]:
cancelled = df[df['status'] == 'cancelled']
print(cancelled.groupby('_true_label')[['displacement_km', 'km_after_cancel']].agg(['min', 'mean', 'max']))

                    displacement_km                       km_after_cancel  \
                                min       mean        max             min   
_true_label                                                                 
fraud_cancel_naive         2.379362  11.249273  20.845882        2.378481   
fraud_cancel_silent        3.481600  10.961070  21.476818        3.480091   
genuine_cancel             0.757363   2.705916   5.826626        0.757712   

                                          
                         mean        max  
_true_label                               
fraud_cancel_naive   9.644143  20.371013  
fraud_cancel_silent  8.981220  17.587770  
genuine_cancel       2.705805   5.818124  


In [158]:
df['flagged'] = (
    (df['status'] == 'cancelled') &
    ((df['displacement_km'] > 0.2) | (df['km_after_cancel'] > 0.2))
)

In [159]:
df['is_fraud'] = df['_true_label'].str.startswith('fraud')

In [160]:
df['km_after_cancel'] = df.apply(km_after_cancel, axis=1)

cancelled = df[df['status'] == 'cancelled']
print(cancelled.groupby('_true_label')[['displacement_km', 'km_after_cancel']].agg(['min', 'mean', 'max']))

df['flagged'] = (
    (df['status'] == 'cancelled') &
    ((df['displacement_km'] > 0.2) | (df['km_after_cancel'] > 0.2))
)

df['is_fraud'] = df['_true_label'].str.startswith('fraud')
print(pd.crosstab(df['is_fraud'], df['flagged']))

                    displacement_km                       km_after_cancel  \
                                min       mean        max             min   
_true_label                                                                 
fraud_cancel_naive         2.379362  11.249273  20.845882        2.378481   
fraud_cancel_silent        3.481600  10.961070  21.476818        3.480091   
genuine_cancel             0.757363   2.705916   5.826626        0.757712   

                                          
                         mean        max  
_true_label                               
fraud_cancel_naive   9.644143  20.371013  
fraud_cancel_silent  8.981220  17.587770  
genuine_cancel       2.705805   5.818124  
flagged   False  True 
is_fraud              
False      2164    256
True          0     66


In [161]:
df['is_fraud'] = df['_true_label'].str.startswith('fraud')
by_driver = df.groupby('driver_id').agg(
    trips=('trip_id', 'count'),
    cancels=('status', lambda s: (s == 'cancelled').sum()),
    frauds=('is_fraud', 'sum')
)
by_driver['cancel_rate'] = by_driver['cancels'] / by_driver['trips']
print(by_driver.sort_values('cancel_rate', ascending=False).head(10))

           trips  cancels  frauds  cancel_rate
driver_id                                     
26            48       22      19     0.458333
10            49       21      17     0.428571
33            45       16       9     0.355556
5             47       16      13     0.340426
49            53       15       8     0.283019
41            56       10       0     0.178571
7             52        9       0     0.173077
34            49        8       0     0.163265
22            51        8       0     0.156863
28            58        9       0     0.155172


In [162]:
p0 = (df['status'] == 'cancelled').mean()
print(round(p0, 3))

by_driver['expected'] = by_driver['trips'] * p0
by_driver['sd'] = (by_driver['trips'] * p0 * (1 - p0)) ** 0.5
by_driver['z'] = (by_driver['cancels'] - by_driver['expected']) / by_driver['sd']
print(by_driver.sort_values('z', ascending=False).head(8)[['trips', 'cancels', 'frauds', 'z']])

0.13
           trips  cancels  frauds         z
driver_id                                  
26            48       22      19  6.784339
10            49       21      17  6.234200
33            45       16       9  4.515619
5             47       16      13  4.305965
49            53       15       8  3.327918
41            56       10       0  1.093057
7             52        9       0  0.935298
34            49        8       0  0.703375


In [163]:
def speed_before_cancel(row):
    if pd.isna(row['cancel_time']):
        return None
    start = row['cancel_time'] - pd.Timedelta(minutes=1)
    window = [p for p in row['gps_trace'] if start <= p['time'] <= row['cancel_time']]
    if len(window) < 2:
        return None
    hours = (window[-1]['time'] - window[0]['time']).total_seconds() / 3600
    return distance_km(window[0], window[-1]) / hours

df['speed_before_cancel'] = df.apply(speed_before_cancel, axis=1)
cancelled = df[df['status'] == 'cancelled']
print(cancelled.groupby('_true_label')['speed_before_cancel'].describe()[['count', 'min', '50%', 'max']])

                     count       min       50%        max
_true_label                                              
fraud_cancel_naive    40.0  0.106867  1.388119  39.984585
fraud_cancel_silent   26.0  0.451478  1.589631  35.175541
genuine_cancel       256.0  0.144448  1.379088   3.036283


In [164]:
cancelled = df[df['status'] == 'cancelled'].copy()
cancelled['flag_speed'] = cancelled['speed_before_cancel'] > 5
cancelled['is_fraud'] = cancelled['_true_label'].str.startswith('fraud')
print(pd.crosstab(cancelled['is_fraud'], cancelled['flag_speed']))

flag_speed  False  True 
is_fraud                
False         256      0
True           36     30


In [165]:
df = df.sort_values(['driver_id', 'pickup_time']).reset_index(drop=True)
df['next_pickup'] = df.groupby('driver_id')['pickup_time'].shift(-1)
df['idle_after_cancel'] = (df['next_pickup'] - df['cancel_time']).dt.total_seconds() / 60
same_day = df['next_pickup'].dt.date == df['pickup_time'].dt.date
df.loc[~same_day, 'idle_after_cancel'] = None

In [166]:
cancelled = df[df['status'] == 'cancelled']
print(cancelled.groupby('_true_label')['idle_after_cancel'].describe()[['count', 'min', '50%', 'max']])

                     count   min   50%    max
_true_label                                  
fraud_cancel_naive    39.0   7.0  32.0   97.0
fraud_cancel_silent   23.0  16.0  32.0  115.0
genuine_cancel       247.0   2.0  10.0  102.0


In [167]:
cancelled = df[df['status'] == 'cancelled'].copy()
cancelled['is_fraud'] = cancelled['_true_label'].str.startswith('fraud')
cancelled['flag_speed'] = cancelled['speed_before_cancel'] > 5
cancelled['flag_idle'] = cancelled['idle_after_cancel'] > 20
cancelled['flag_any'] = cancelled['flag_speed'] | cancelled['flag_idle']

for col in ['flag_speed', 'flag_idle', 'flag_any']:
    caught = (cancelled['is_fraud'] & cancelled[col]).sum()
    missed = (cancelled['is_fraud'] & ~cancelled[col]).sum()
    false_alarms = (~cancelled['is_fraud'] & cancelled[col]).sum()
    print(col, 'caught', caught, 'missed', missed, 'false alarms', false_alarms)

flag_speed caught 30 missed 36 false alarms 0
flag_idle caught 54 missed 12 false alarms 20
flag_any caught 57 missed 9 false alarms 20


In [168]:
by_driver = cancelled.groupby('driver_id').agg(
    cancels=('trip_id', 'count'),
    flags=('flag_any', 'sum'),
    frauds=('is_fraud', 'sum')
)
by_driver['flag_rate'] = by_driver['flags'] / by_driver['cancels']
print(by_driver.sort_values('flags', ascending=False).head(12))

           cancels  flags  frauds  flag_rate
driver_id                                   
26              22     18      19   0.818182
10              21     14      17   0.666667
5               16     11      13   0.687500
33              16      9       9   0.562500
49              15      7       8   0.466667
9                3      2       0   0.666667
18               4      1       0   0.250000
45               8      1       0   0.125000
43               5      1       0   0.200000
39               5      1       0   0.200000
38               4      1       0   0.250000
37               7      1       0   0.142857


In [169]:
ranked = by_driver.sort_values('flags', ascending=False)
honest_like = ranked.iloc[int(len(ranked) * 0.2):]
p0 = honest_like['flags'].sum() / honest_like['cancels'].sum()
print(round(p0, 3))

by_driver['p_value'] = binom.sf(by_driver['flags'] - 1, by_driver['cancels'], p0)
alpha = 0.05 / len(by_driver)
by_driver['flagged_driver'] = by_driver['p_value'] < alpha
print(alpha)


0.058
0.001


In [170]:
by_driver['is_fraudster'] = by_driver['frauds'] > 0
print(pd.crosstab(by_driver['is_fraudster'], by_driver['flagged_driver']))

flagged_driver  False  True 
is_fraudster                
False              45      0
True                0      5


In [171]:
from scipy.stats import binom

def run_detection(df, speed_limit=5, idle_limit=20):
    df = df.sort_values(['driver_id', 'pickup_time']).reset_index(drop=True)
    df['next_pickup'] = df.groupby('driver_id')['pickup_time'].shift(-1)
    df['idle_after_cancel'] = (df['next_pickup'] - df['cancel_time']).dt.total_seconds() / 60
    same_day = df['next_pickup'].dt.date == df['pickup_time'].dt.date
    df.loc[~same_day, 'idle_after_cancel'] = None
    df['speed_before_cancel'] = df.apply(speed_before_cancel, axis=1)

    cancelled = df[df['status'] == 'cancelled'].copy()
    cancelled['flag_any'] = ((cancelled['speed_before_cancel'] > speed_limit) |
                             (cancelled['idle_after_cancel'] > idle_limit))
    cancelled['is_fraud'] = cancelled['_true_label'].str.startswith('fraud')

    by_driver = cancelled.groupby('driver_id').agg(
        cancels=('trip_id', 'count'),
        flags=('flag_any', 'sum'),
        frauds=('is_fraud', 'sum')
    )
    ranked = by_driver.sort_values('flags', ascending=False)
    honest_like = ranked.iloc[int(len(ranked) * 0.2):]
    p0 = honest_like['flags'].sum() / honest_like['cancels'].sum()

    by_driver['p_value'] = binom.sf(by_driver['flags'] - 1, by_driver['cancels'], p0)
    by_driver['flagged_driver'] = by_driver['p_value'] < 0.05 / len(by_driver)
    by_driver['is_fraudster'] = by_driver['frauds'] > 0
    return by_driver

In [172]:
for name in ['subtle_2d.pkl', 'subtle_7d.pkl']:
    result = run_detection(pd.read_pickle(name))
    caught = (result['is_fraudster'] & result['flagged_driver']).sum()
    n_fraud = result['is_fraudster'].sum()
    wrong = (~result['is_fraudster'] & result['flagged_driver']).sum()
    print(name, 'caught', caught, 'of', n_fraud, '| honest drivers accused:', wrong)

subtle_2d.pkl caught 3 of 5 | honest drivers accused: 0
subtle_7d.pkl caught 5 of 5 | honest drivers accused: 0
